# Datathon Passos Mágicos — Tratamento e Padronização dos Dados

## 1. Introdução

Este notebook dá continuidade à análise exploratória realizada no `01_exploracao.ipynb`.

A etapa anterior mostrou que as bases de 2022, 2023 e 2024 possuem estruturas e nomenclaturas diferentes, além de inconsistências de tipo e preenchimento. O objetivo agora é transformar essas fontes em uma **base longitudinal padronizada**, na qual cada linha representa um aluno em determinado ano.

A base resultante será utilizada nas próximas etapas do projeto:

- análise dos indicadores educacionais;
- construção do dashboard no Power BI;
- preparação do modelo preditivo;
- aplicação no Streamlit.

### Problemas identificados na exploração

Entre os principais pontos encontrados no notebook anterior estão:

- nomes diferentes para a mesma variável entre os anos;
- `Idade` de 2023 parcialmente interpretada como data pelo Excel;
- `INDE 2024` e `Pedra 2024` contendo o valor textual `INCLUIR`;
- ausência de `IPP` em 2022;
- categorias diferentes para gênero e Pedra;
- diversas colunas totalmente nulas em 2023 e 2024;
- codificações diferentes para `Fase` entre os anos;
- necessidade de validar o significado da variável `Defasagem` antes da modelagem.

> **Princípio adotado:** não será feita imputação estatística neste notebook. Valores verdadeiramente ausentes serão preservados como nulos. A imputação necessária para Machine Learning será feita posteriormente dentro do pipeline de treino, evitando vazamento de informação.

In [18]:
import re
from datetime import datetime, date
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


## 2. Localização dos arquivos

O caminho é construído de forma relativa para que o projeto continue funcionando após ser clonado do GitHub, sem depender do diretório específico de um computador.

A estrutura esperada é:

```text
DATATHON 5/
├── data/
│   ├── raw/
│   └── processed/
└── notebooks/
    ├── 01_exploracao.ipynb
    └── 02_tratamento.ipynb
```

In [19]:
NOME_ARQUIVO = "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

possiveis_raizes = [Path.cwd().resolve(), Path.cwd().resolve().parent]

PROJECT_ROOT = None
for raiz in possiveis_raizes:
    candidato = raiz / "data" / "raw" / NOME_ARQUIVO
    if candidato.exists():
        PROJECT_ROOT = raiz
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Arquivo não encontrado em data/raw. Verifique a estrutura do projeto."
    )

DATA_PATH = PROJECT_ROOT / "data" / "raw" / NOME_ARQUIVO
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Base original: {DATA_PATH}")
print(f"Saída tratada: {PROCESSED_PATH}")

Raiz do projeto: C:\Users\chopa\OneDrive\Área de Trabalho\DATATHON 5
Base original: C:\Users\chopa\OneDrive\Área de Trabalho\DATATHON 5\data\raw\BASE DE DADOS PEDE 2024 - DATATHON.xlsx
Saída tratada: C:\Users\chopa\OneDrive\Área de Trabalho\DATATHON 5\data\processed


## 3. Carregamento das bases originais

As três abas são carregadas separadamente. Nesta etapa ainda não alteramos os dados originais; cada transformação será aplicada em novas estruturas.

In [20]:
df_2022 = pd.read_excel(DATA_PATH, sheet_name="PEDE2022")
df_2023 = pd.read_excel(DATA_PATH, sheet_name="PEDE2023")
df_2024 = pd.read_excel(DATA_PATH, sheet_name="PEDE2024")

bases_raw = {
    2022: df_2022,
    2023: df_2023,
    2024: df_2024,
}

for ano, df in bases_raw.items():
    print(f"{ano}: {df.shape[0]:,} registros x {df.shape[1]} colunas")

2022: 860 registros x 42 colunas
2023: 1,014 registros x 48 colunas
2024: 1,156 registros x 50 colunas


## 4. Regras de padronização

Em vez de renomear todas as dezenas de colunas das três planilhas, será construída uma **seleção curada das variáveis relevantes** para análise e modelagem.

Essa estratégia também evita carregar para a base longitudinal:

- colunas históricas repetidas, como `Pedra 20`, `Pedra 21`, `INDE 22` em anos posteriores;
- campos operacionais totalmente vazios;
- colunas de avaliadores que não são necessárias para responder ao case;
- duplicidades de nomes criadas no arquivo original.

Serão preservados os campos educacionais, demográficos e de contexto úteis para o projeto.

In [21]:
MAPAS_COLUNAS = {
    2022: {
        "ra": "RA",
        "fase_original": "Fase",
        "turma": "Turma",
        "nome": "Nome",
        "data_nascimento": None,
        "ano_nascimento": "Ano nasc",
        "idade": "Idade 22",
        "genero": "Gênero",
        "ano_ingresso": "Ano ingresso",
        "instituicao_ensino": "Instituição de ensino",
        "escola": None,
        "status_aluno": None,
        "pedra": "Pedra 22",
        "inde": "INDE 22",
        "iaa": "IAA",
        "ieg": "IEG",
        "ips": "IPS",
        "ipp": None,
        "ida": "IDA",
        "nota_matematica": "Matem",
        "nota_portugues": "Portug",
        "nota_ingles": "Inglês",
        "indicado_bolsa": "Indicado",
        "atingiu_pv": "Atingiu PV",
        "ipv": "IPV",
        "ian": "IAN",
        "fase_ideal": "Fase ideal",
        "defasagem_original": "Defas",
        "rec_psicologia": "Rec Psicologia",
        "destaque_ieg": "Destaque IEG",
        "destaque_ida": "Destaque IDA",
        "destaque_ipv": "Destaque IPV",
    },
    2023: {
        "ra": "RA",
        "fase_original": "Fase",
        "turma": "Turma",
        "nome": "Nome Anonimizado",
        "data_nascimento": "Data de Nasc",
        "ano_nascimento": None,
        "idade": "Idade",
        "genero": "Gênero",
        "ano_ingresso": "Ano ingresso",
        "instituicao_ensino": "Instituição de ensino",
        "escola": None,
        "status_aluno": None,
        "pedra": "Pedra 2023",
        "inde": "INDE 2023",
        "iaa": "IAA",
        "ieg": "IEG",
        "ips": "IPS",
        "ipp": "IPP",
        "ida": "IDA",
        "nota_matematica": "Mat",
        "nota_portugues": "Por",
        "nota_ingles": "Ing",
        "indicado_bolsa": "Indicado",
        "atingiu_pv": "Atingiu PV",
        "ipv": "IPV",
        "ian": "IAN",
        "fase_ideal": "Fase Ideal",
        "defasagem_original": "Defasagem",
        "rec_psicologia": "Rec Psicologia",
        "destaque_ieg": "Destaque IEG",
        "destaque_ida": "Destaque IDA",
        "destaque_ipv": "Destaque IPV",
    },
    2024: {
        "ra": "RA",
        "fase_original": "Fase",
        "turma": "Turma",
        "nome": "Nome Anonimizado",
        "data_nascimento": "Data de Nasc",
        "ano_nascimento": None,
        "idade": "Idade",
        "genero": "Gênero",
        "ano_ingresso": "Ano ingresso",
        "instituicao_ensino": "Instituição de ensino",
        "escola": "Escola",
        "status_aluno": "Ativo/ Inativo",
        "pedra": "Pedra 2024",
        "inde": "INDE 2024",
        "iaa": "IAA",
        "ieg": "IEG",
        "ips": "IPS",
        "ipp": "IPP",
        "ida": "IDA",
        "nota_matematica": "Mat",
        "nota_portugues": "Por",
        "nota_ingles": "Ing",
        "indicado_bolsa": "Indicado",
        "atingiu_pv": "Atingiu PV",
        "ipv": "IPV",
        "ian": "IAN",
        "fase_ideal": "Fase Ideal",
        "defasagem_original": "Defasagem",
        "rec_psicologia": "Rec Psicologia",
        "destaque_ieg": "Destaque IEG",
        "destaque_ida": "Destaque IDA",
        "destaque_ipv": "Destaque IPV",
    },
}

## 5. Funções auxiliares de limpeza

As funções abaixo concentram as principais regras de tratamento.

### Decisões importantes

**Idade de 2023:** alguns valores numéricos foram formatados no Excel como datas de janeiro de 1900. Por exemplo, uma idade `8` aparece como `1900-01-08`. Nesses casos, o dia da data corresponde ao valor numérico original.

**Pedra:** somente as quatro classificações previstas no dataset serão mantidas: `Quartzo`, `Ágata`, `Ametista` e `Topázio`. O texto `INCLUIR`, presente em 2024, não representa uma classificação e será convertido em nulo.

**Fase:** a codificação muda ao longo do período. Em 2022 ela é numérica, em 2023 aparece como `FASE 1`, `FASE 2` etc., e em 2024 há códigos como `3A`, `7E` e `8B`. Será criada uma variável numérica `fase_ordem`, preservando também o valor original.

In [22]:
def corrigir_idade(valor):
    """Corrige idades numéricas que foram interpretadas pelo Excel como datas."""
    if pd.isna(valor):
        return np.nan

    if isinstance(valor, (pd.Timestamp, datetime, date)):
        return float(valor.day)

    try:
        return float(valor)
    except (TypeError, ValueError):
        return np.nan


def converter_data_excel(valor):
    """Converte strings, datas ou números seriais do Excel para datetime."""
    if pd.isna(valor):
        return pd.NaT

    if isinstance(valor, (pd.Timestamp, datetime, date)):
        return pd.Timestamp(valor)

    if isinstance(valor, (int, float, np.integer, np.floating)) and valor > 1000:
        return pd.to_datetime(
            valor,
            unit="D",
            origin="1899-12-30",
            errors="coerce",
        )

    return pd.to_datetime(valor, errors="coerce", dayfirst=False)


def extrair_fase_ordem(valor):
    """Transforma ALFA em 0 e extrai a parte numérica das demais fases."""
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().upper()

    if "ALFA" in texto:
        return 0.0

    numero = re.search(r"(\d+)", texto)
    return float(numero.group(1)) if numero else np.nan


def extrair_fase_ideal_ordem(valor):
    """Extrai a ordem da descrição de Fase Ideal."""
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().upper()

    if "ALFA" in texto:
        return 0.0

    numero = re.search(r"FASE\s*(\d+)", texto)
    return float(numero.group(1)) if numero else np.nan


def nome_padronizado_fase(ordem):
    if pd.isna(ordem):
        return pd.NA

    ordem = int(ordem)
    return "ALFA" if ordem == 0 else f"FASE {ordem}"


def padronizar_genero(valor):
    if pd.isna(valor):
        return pd.NA

    texto = str(valor).strip().lower()

    mapa = {
        "menina": "Feminino",
        "feminino": "Feminino",
        "menino": "Masculino",
        "masculino": "Masculino",
    }

    return mapa.get(texto, str(valor).strip())


def padronizar_pedra(valor):
    if pd.isna(valor):
        return pd.NA

    texto = str(valor).strip()

    mapa = {
        "Quartzo": "Quartzo",
        "Agata": "Ágata",
        "Ágata": "Ágata",
        "Ametista": "Ametista",
        "Topazio": "Topázio",
        "Topázio": "Topázio",
    }

    # Valores fora das quatro categorias, como "INCLUIR", ficam ausentes.
    return mapa.get(texto, pd.NA)


def padronizar_instituicao(valor):
    if pd.isna(valor):
        return pd.NA

    texto = " ".join(str(valor).strip().split())

    mapa = {
        "Escola Pública": "Pública",
        "Privada - Programa de apadrinhamento": "Privada - Programa de Apadrinhamento",
    }

    return mapa.get(texto, texto)


def padronizar_sim_nao(valor):
    if pd.isna(valor):
        return pd.NA

    texto = str(valor).strip().lower()

    if texto in {"sim", "s", "yes", "1", "true"}:
        return "Sim"
    if texto in {"não", "nao", "n", "no", "0", "false"}:
        return "Não"

    return str(valor).strip()

## 6. Correção específica da idade de 2023

Antes de construir a base final, vamos quantificar a inconsistência e verificar o resultado da regra de correção.

In [23]:
tipos_idade_2023 = df_2023["Idade"].map(lambda x: type(x).__name__).value_counts()
print("Tipos encontrados antes da correção:")
display(tipos_idade_2023.to_frame("Quantidade"))

idade_2023_corrigida = df_2023["Idade"].map(corrigir_idade)

print("\nApós a correção:")
print(f"Nulos: {idade_2023_corrigida.isna().sum()}")
print(f"Mínimo: {idade_2023_corrigida.min():.0f}")
print(f"Máximo: {idade_2023_corrigida.max():.0f}")
print(f"Média: {idade_2023_corrigida.mean():.2f}")

Tipos encontrados antes da correção:


,Quantidade
Idade,
int,615
datetime,399



Após a correção:
Nulos: 0
Mínimo: 7
Máximo: 26
Média: 12.33


## 7. Construção das bases anuais padronizadas

Cada base anual será convertida para o mesmo esquema de colunas. Variáveis inexistentes em determinado ano serão mantidas como nulas, em vez de serem preenchidas artificialmente.

Isso é especialmente importante para o `IPP`, que não está disponível em 2022.

In [24]:
def construir_base_ano(df, ano):
    mapa = MAPAS_COLUNAS[ano]
    resultado = pd.DataFrame(index=df.index)

    resultado["ano"] = ano

    for coluna_final, coluna_original in mapa.items():
        if coluna_original is None:
            resultado[coluna_final] = pd.NA
        else:
            resultado[coluna_final] = df[coluna_original]

    # Datas e ano de nascimento
    resultado["data_nascimento"] = resultado["data_nascimento"].map(converter_data_excel)

    if ano == 2022:
        resultado["ano_nascimento"] = pd.to_numeric(
            resultado["ano_nascimento"], errors="coerce"
        )
    else:
        resultado["ano_nascimento"] = resultado["data_nascimento"].dt.year

    # Correções e padronizações categóricas
    resultado["idade"] = resultado["idade"].map(corrigir_idade)
    resultado["genero"] = resultado["genero"].map(padronizar_genero)
    resultado["instituicao_ensino"] = resultado["instituicao_ensino"].map(
        padronizar_instituicao
    )
    resultado["pedra"] = resultado["pedra"].map(padronizar_pedra)
    resultado["indicado_bolsa"] = resultado["indicado_bolsa"].map(padronizar_sim_nao)
    resultado["atingiu_pv"] = resultado["atingiu_pv"].map(padronizar_sim_nao)

    # Conversão explícita das métricas numéricas
    colunas_numericas = [
        "ano_ingresso",
        "inde",
        "iaa",
        "ieg",
        "ips",
        "ipp",
        "ida",
        "nota_matematica",
        "nota_portugues",
        "nota_ingles",
        "ipv",
        "ian",
        "defasagem_original",
    ]

    for coluna in colunas_numericas:
        resultado[coluna] = pd.to_numeric(resultado[coluna], errors="coerce")

    # Padronização das fases
    resultado["fase_ordem"] = resultado["fase_original"].map(extrair_fase_ordem)
    resultado["fase"] = resultado["fase_ordem"].map(nome_padronizado_fase)

    resultado["fase_ideal_ordem"] = resultado["fase_ideal"].map(
        extrair_fase_ideal_ordem
    )
    resultado["fase_ideal_padronizada"] = resultado["fase_ideal_ordem"].map(
        nome_padronizado_fase
    )

    # A defasagem observada no dataset corresponde a Fase Atual - Fase Ideal.
    resultado["defasagem_calculada"] = (
        resultado["fase_ordem"] - resultado["fase_ideal_ordem"]
    )

    resultado["defasagem_inconsistente"] = (
        resultado["defasagem_original"].notna()
        & resultado["defasagem_calculada"].notna()
        & (
            resultado["defasagem_original"]
            != resultado["defasagem_calculada"]
        )
    )

    # Quando Fase e Fase Ideal estão disponíveis, usamos a relação matemática
    # validada na própria base. O valor original continua preservado para auditoria.
    resultado["defasagem"] = resultado["defasagem_calculada"].combine_first(
        resultado["defasagem_original"]
    )

    resultado["status_defasagem"] = np.select(
        [
            resultado["defasagem"] < 0,
            resultado["defasagem"] == 0,
            resultado["defasagem"] > 0,
        ],
        ["Defasado", "Adequado", "Adiantado"],
        default=None,
    )

    resultado["em_defasagem"] = np.where(
        resultado["defasagem"].isna(),
        np.nan,
        (resultado["defasagem"] < 0).astype(int),
    )

    return resultado


base_2022 = construir_base_ano(df_2022, 2022)
base_2023 = construir_base_ano(df_2023, 2023)
base_2024 = construir_base_ano(df_2024, 2024)

print(base_2022.shape, base_2023.shape, base_2024.shape)

(860, 42) (1014, 42) (1156, 42)


## 8. Validação da variável Defasagem

A exploração anterior mostrou que era necessário compreender o sinal da variável `Defasagem`.

A padronização das fases permite testar diretamente a relação:

\[
\text{Defasagem} = \text{Fase Atual} - \text{Fase Ideal}
\]

Portanto:

- valor **negativo**: aluno está em uma fase abaixo da fase ideal → `Defasado`;
- valor **zero**: aluno está na fase esperada → `Adequado`;
- valor **positivo**: aluno está acima da fase ideal → `Adiantado`.

A célula seguinte verifica essa regra em todos os registros e identifica eventuais inconsistências do arquivo original.

In [25]:
validacao_defasagem = []

for ano, df in {
    2022: base_2022,
    2023: base_2023,
    2024: base_2024,
}.items():
    comparaveis = (
        df["defasagem_original"].notna()
        & df["defasagem_calculada"].notna()
    )

    iguais = (
        df.loc[comparaveis, "defasagem_original"]
        == df.loc[comparaveis, "defasagem_calculada"]
    )

    validacao_defasagem.append({
        "Ano": ano,
        "Registros comparáveis": int(comparaveis.sum()),
        "Coincidentes": int(iguais.sum()),
        "Inconsistentes": int((~iguais).sum()),
        "% consistência": round(iguais.mean() * 100, 2),
    })

validacao_defasagem = pd.DataFrame(validacao_defasagem)
display(validacao_defasagem)

,Ano,Registros comparáveis,Coincidentes,Inconsistentes,% consistência
0,2022,860,860,0,100.00
1,2023,1014,1014,0,100.00
2,2024,1156,1154,2,99.83


In [26]:
inconsistencias_defasagem = pd.concat(
    [base_2022, base_2023, base_2024],
    ignore_index=True,
).loc[
    lambda x: x["defasagem_inconsistente"],
    [
        "ra",
        "ano",
        "fase_original",
        "fase_ideal",
        "defasagem_original",
        "defasagem_calculada",
        "defasagem",
    ],
]

print(f"Inconsistências encontradas: {len(inconsistencias_defasagem)}")
display(inconsistencias_defasagem)

Inconsistências encontradas: 2


,ra,ano,fase_original,fase_ideal,defasagem_original,defasagem_calculada,defasagem
2444,RA-1516,2024,3A,Fase 3 (7° e 8° ano),3,0.0,0.0
2452,RA-1519,2024,3A,Fase 3 (7° e 8° ano),3,0.0,0.0


### Tratamento adotado para as inconsistências de defasagem

A relação entre fase atual e fase ideal é consistente em praticamente toda a base. Apenas registros isolados divergem da regra matemática observada nos demais alunos.

Por isso:

- `defasagem_original` preserva exatamente o valor recebido;
- `defasagem_calculada` registra `fase_ordem - fase_ideal_ordem`;
- `defasagem` é a versão tratada utilizada nas próximas análises;
- `defasagem_inconsistente` permite rastrear os registros corrigidos.

Esse desenho mantém a **auditabilidade** do tratamento.

## 9. Consolidação longitudinal

As três bases padronizadas serão empilhadas verticalmente. A chave lógica passa a ser o par:

```text
RA + Ano
```

Assim, o mesmo aluno pode aparecer em vários anos sem representar duplicidade.

In [27]:
base_tratada = pd.concat(
    [base_2022, base_2023, base_2024],
    ignore_index=True,
)

ordem_colunas = [
    "ra",
    "ano",
    "nome",
    "ano_nascimento",
    "data_nascimento",
    "idade",
    "genero",
    "ano_ingresso",
    "instituicao_ensino",
    "escola",
    "status_aluno",
    "fase",
    "fase_ordem",
    "fase_original",
    "turma",
    "fase_ideal_padronizada",
    "fase_ideal_ordem",
    "fase_ideal",
    "defasagem_original",
    "defasagem_calculada",
    "defasagem_inconsistente",
    "defasagem",
    "status_defasagem",
    "em_defasagem",
    "pedra",
    "inde",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
    "indicado_bolsa",
    "atingiu_pv",
    "rec_psicologia",
    "destaque_ieg",
    "destaque_ida",
    "destaque_ipv",
]

base_tratada = base_tratada[ordem_colunas]

print(f"Base longitudinal: {base_tratada.shape[0]:,} linhas x {base_tratada.shape[1]} colunas")
print(f"Duplicidades RA + ano: {base_tratada.duplicated(['ra', 'ano']).sum()}")

display(base_tratada.head())

Base longitudinal: 3,030 linhas x 42 colunas
Duplicidades RA + ano: 0


,ra,ano,nome,ano_nascimento,data_nascimento,idade,genero,ano_ingresso,instituicao_ensino,escola,status_aluno,fase,fase_ordem,fase_original,turma,fase_ideal_padronizada,fase_ideal_ordem,fase_ideal,defasagem_original,defasagem_calculada,defasagem_inconsistente,defasagem,status_defasagem,em_defasagem,pedra,inde,ian,ida,ieg,iaa,ips,ipp,ipv,nota_matematica,nota_portugues,nota_ingles,indicado_bolsa,atingiu_pv,rec_psicologia,destaque_ieg,destaque_ida,destaque_ipv
0,RA-1,2022,Aluno-1,2003,NaT,19.0,Feminino,2016,Pública,NaN,NaN,FASE 7,7.0,7,A,FASE 8,8.0,Fase 8 (Universitários),-1,-1.0,False,-1.0,Defasado,1.0,Quartzo,5.783,5.0,4.0,4.1,8.3,5.6,NaN,7.278,2.7,3.5,6.0,Sim,Não,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.
1,RA-2,2022,Aluno-2,2005,NaT,17.0,Feminino,2017,Rede Decisão,NaN,NaN,FASE 7,7.0,7,A,FASE 7,7.0,Fase 7 (3º EM),0,0.0,False,0.0,Adequado,0.0,Ametista,7.055,10.0,6.8,5.2,8.8,6.3,NaN,6.778,6.3,4.5,9.7,Não,Não,Sem limitações,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.
2,RA-3,2022,Aluno-3,2005,NaT,17.0,Feminino,2016,Rede Decisão,NaN,NaN,FASE 7,7.0,7,A,FASE 7,7.0,Fase 7 (3º EM),0,0.0,False,0.0,Adequado,0.0,Ágata,6.591,10.0,5.6,7.9,0.0,5.6,NaN,7.556,5.8,4.0,6.9,Não,Não,Sem limitações,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Destaque: A sua boa integração aos Princípios Passos Mágicos.
3,RA-4,2022,Aluno-4,2005,NaT,17.0,Masculino,2017,Rede Decisão,NaN,NaN,FASE 7,7.0,7,A,FASE 7,7.0,Fase 7 (3º EM),0,0.0,False,0.0,Adequado,0.0,Quartzo,5.951,10.0,5.0,4.5,8.8,5.6,NaN,5.278,2.8,3.5,8.7,Não,Não,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.
4,RA-5,2022,Aluno-5,2005,NaT,17.0,Feminino,2016,Rede Decisão,NaN,NaN,FASE 7,7.0,7,A,FASE 7,7.0,Fase 7 (3º EM),0,0.0,False,0.0,Adequado,0.0,Ametista,7.427,10.0,5.2,8.6,7.9,5.6,NaN,7.389,7.0,2.9,5.7,Não,Não,Requer avaliação,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliações.,Melhorar: Integrar-se mais aos Princípios Passos Mágicos.


## 10. Validações após o tratamento

Agora verificamos os principais pontos que motivaram a limpeza:

- idade;
- categorias de gênero;
- Pedra;
- indicadores numéricos;
- fases;
- duplicidades;
- defasagem.

In [28]:
resumo_validacao = []

for ano, df in base_tratada.groupby("ano"):
    resumo_validacao.append({
        "Ano": ano,
        "Registros": len(df),
        "RA únicos": df["ra"].nunique(),
        "Duplicados RA+Ano": df.duplicated(["ra", "ano"]).sum(),
        "Idade nula": df["idade"].isna().sum(),
        "Fase não interpretada": df["fase_ordem"].isna().sum(),
        "Fase ideal não interpretada": df["fase_ideal_ordem"].isna().sum(),
        "INDE nulo": df["inde"].isna().sum(),
        "Pedra nula": df["pedra"].isna().sum(),
        "Defasagem nula": df["defasagem"].isna().sum(),
    })

resumo_validacao = pd.DataFrame(resumo_validacao)
display(resumo_validacao)

,Ano,Registros,RA únicos,Duplicados RA+Ano,Idade nula,Fase não interpretada,Fase ideal não interpretada,INDE nulo,Pedra nula,Defasagem nula
0,2022,860,860,0,0,0,0,0,0,0
1,2023,1014,1014,0,0,0,0,83,83,0
2,2024,1156,1156,0,0,0,0,102,102,0


In [29]:
print("Categorias de gênero após padronização:")
display(
    pd.crosstab(
        base_tratada["ano"],
        base_tratada["genero"],
        dropna=False,
    )
)

print("\nCategorias de Pedra após padronização:")
display(
    pd.crosstab(
        base_tratada["ano"],
        base_tratada["pedra"],
        dropna=False,
    )
)

Categorias de gênero após padronização:


genero,Feminino,Masculino
ano,,
2022,457,403
2023,546,468
2024,623,533



Categorias de Pedra após padronização:


pedra,Ametista,Quartzo,Topázio,Ágata,NaN
ano,,,,,
2022,348,132,130,250,0
2023,381,72,232,246,83
2024,391,112,326,225,102


In [30]:
print("Distribuição do status de defasagem após a correção:")

display(
    pd.crosstab(
        base_tratada["ano"],
        base_tratada["status_defasagem"],
        dropna=False,
    )
)

Distribuição do status de defasagem após a correção:


status_defasagem,Adequado,Adiantado,Defasado
ano,,,
2022,247,12,601
2023,420,42,552
2024,487,135,534


## 11. Valores ausentes após a padronização

Nulos não serão preenchidos automaticamente nesta etapa.

Há diferenças reais de disponibilidade entre os anos, como:

- `IPP` inexistente em 2022;
- nota de Inglês ausente para grande parte dos alunos;
- indicadores incompletos em parte de 2023 e 2024;
- campos textuais de acompanhamento disponíveis principalmente em 2022.

Substituir esses valores por zero alteraria seu significado. Por isso, eles permanecem como `NaN`/`NA`.

In [31]:
colunas_analiticas = [
    "inde",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
]

nulos_indicadores = (
    base_tratada
    .groupby("ano")[colunas_analiticas]
    .agg(lambda x: x.isna().sum())
    .T
)

nulos_indicadores.columns.name = None
nulos_indicadores.index.name = "Variável"

display(nulos_indicadores)

,2022,2023,2024
Variável,,,
inde,0,83,102
ian,0,0,0
ida,0,77,101
ieg,0,76,0
iaa,0,63,102
ips,0,69,102
ipp,860,76,102
ipv,0,76,102
nota_matematica,2,77,105


## 12. Preparação de uma base temporal para Machine Learning

O enunciado do Datathon pede um modelo capaz de estimar o risco de defasagem. Para reduzir o risco de *data leakage*, a estrutura mais interessante é usar informações de um ano para prever a situação do aluno no ano seguinte.

Como existem **765 alunos presentes simultaneamente em 2023 e 2024**, será criada uma base auxiliar com:

```text
Indicadores de 2023  →  Situação de defasagem em 2024
```

Nesta etapa **nenhum modelo será treinado**. O objetivo é apenas criar a estrutura que será utilizada no notebook de Machine Learning.

Também serão mantidas duas possibilidades de alvo:

1. `alvo_defasagem_2024`: indica se o aluno está defasado em 2024;
2. `entrou_em_defasagem_2024`: para alunos que não estavam defasados em 2023, indica se passaram a ficar defasados em 2024.

A decisão final sobre qual alvo utilizar será tomada no notebook de modelagem após comparar tamanho da amostra, balanceamento e objetivo de negócio.

In [32]:
features_2023 = [
    "ra",
    "fase_ordem",
    "idade",
    "genero",
    "ano_ingresso",
    "instituicao_ensino",
    "inde",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
    "defasagem",
    "em_defasagem",
]

base_features_2023 = base_tratada.loc[
    base_tratada["ano"].eq(2023),
    features_2023,
].copy()

base_features_2023 = base_features_2023.rename(
    columns={
        coluna: f"{coluna}_2023"
        for coluna in base_features_2023.columns
        if coluna != "ra"
    }
)

base_target_2024 = base_tratada.loc[
    base_tratada["ano"].eq(2024),
    ["ra", "defasagem", "em_defasagem"],
].rename(
    columns={
        "defasagem": "defasagem_2024",
        "em_defasagem": "alvo_defasagem_2024",
    }
)

base_modelo_temporal = base_features_2023.merge(
    base_target_2024,
    on="ra",
    how="inner",
    validate="one_to_one",
)

# Alvo alternativo: apenas alunos que estavam adequados/adiantados em 2023.
base_modelo_temporal["elegivel_nova_defasagem_2024"] = (
    base_modelo_temporal["em_defasagem_2023"].eq(0)
).astype(int)

base_modelo_temporal["entrou_em_defasagem_2024"] = np.where(
    base_modelo_temporal["elegivel_nova_defasagem_2024"].eq(1),
    base_modelo_temporal["alvo_defasagem_2024"],
    np.nan,
)

print(f"Base temporal: {base_modelo_temporal.shape[0]} alunos")
print("\nAlvo: aluno defasado em 2024")
display(
    base_modelo_temporal["alvo_defasagem_2024"]
    .value_counts(dropna=False)
    .rename_axis("Classe")
    .to_frame("Quantidade")
)

print("\nAlunos elegíveis para analisar nova entrada em defasagem:")
elegiveis = base_modelo_temporal["elegivel_nova_defasagem_2024"].eq(1)
print(f"Elegíveis: {elegiveis.sum()}")
print(
    "Entraram em defasagem em 2024: "
    f"{int(base_modelo_temporal['entrou_em_defasagem_2024'].sum(skipna=True))}"
)

Base temporal: 765 alunos

Alvo: aluno defasado em 2024


,Quantidade
Classe,
0.0,457
1.0,308



Alunos elegíveis para analisar nova entrada em defasagem:
Elegíveis: 370
Entraram em defasagem em 2024: 84


## 13. Relatório resumido de qualidade

Além da base tratada, será gerado um pequeno arquivo com métricas de qualidade por ano. Ele serve como registro do processo de limpeza e pode ser utilizado posteriormente no README ou na apresentação.

In [33]:
relatorio_qualidade = []

for ano, df in base_tratada.groupby("ano"):
    relatorio_qualidade.append({
        "ano": ano,
        "registros": len(df),
        "alunos_unicos": df["ra"].nunique(),
        "idade_nulos": df["idade"].isna().sum(),
        "inde_nulos": df["inde"].isna().sum(),
        "pedra_nulos": df["pedra"].isna().sum(),
        "ipp_nulos": df["ipp"].isna().sum(),
        "defasagem_nulos": df["defasagem"].isna().sum(),
        "defasagem_inconsistencias_origem": int(df["defasagem_inconsistente"].sum()),
        "alunos_defasados": int(df["em_defasagem"].sum()),
        "percentual_defasados": round(df["em_defasagem"].mean() * 100, 2),
    })

relatorio_qualidade = pd.DataFrame(relatorio_qualidade)
display(relatorio_qualidade)

,ano,registros,alunos_unicos,idade_nulos,inde_nulos,pedra_nulos,ipp_nulos,defasagem_nulos,defasagem_inconsistencias_origem,alunos_defasados,percentual_defasados
0,2022,860,860,0,0,0,860,0,0,601,69.88
1,2023,1014,1014,0,83,83,76,0,0,552,54.44
2,2024,1156,1156,0,102,102,102,0,2,534,46.19


## 14. Exportação dos dados tratados

Serão gerados três arquivos em `data/processed/`:

- `base_tratada.csv`: base longitudinal principal para análise e Power BI;
- `base_modelo_temporal.csv`: estrutura 2023 → 2024 para o notebook de Machine Learning;
- `relatorio_qualidade.csv`: resumo das validações do tratamento.

O CSV principal será salvo com `utf-8-sig`, facilitando a leitura de acentos em ferramentas como Excel e Power BI.

In [34]:
ARQUIVO_BASE_TRATADA = PROCESSED_PATH / "base_tratada.csv"
ARQUIVO_BASE_MODELO = PROCESSED_PATH / "base_modelo_temporal.csv"
ARQUIVO_QUALIDADE = PROCESSED_PATH / "relatorio_qualidade.csv"

base_tratada.to_csv(
    ARQUIVO_BASE_TRATADA,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

base_modelo_temporal.to_csv(
    ARQUIVO_BASE_MODELO,
    index=False,
    encoding="utf-8-sig",
)

relatorio_qualidade.to_csv(
    ARQUIVO_QUALIDADE,
    index=False,
    encoding="utf-8-sig",
)

print("Arquivos gerados com sucesso:")
print(f"- {ARQUIVO_BASE_TRATADA}")
print(f"- {ARQUIVO_BASE_MODELO}")
print(f"- {ARQUIVO_QUALIDADE}")

Arquivos gerados com sucesso:
- C:\Users\chopa\OneDrive\Área de Trabalho\DATATHON 5\data\processed\base_tratada.csv
- C:\Users\chopa\OneDrive\Área de Trabalho\DATATHON 5\data\processed\base_modelo_temporal.csv
- C:\Users\chopa\OneDrive\Área de Trabalho\DATATHON 5\data\processed\relatorio_qualidade.csv


## 15. Conclusões do tratamento

A etapa de limpeza e padronização gerou uma estrutura única e auditável para os três anos do Datathon.

### Resultados do tratamento

- A base longitudinal possui **3.030 registros e 42 colunas**, mantendo um registro por combinação `RA + ano`;
- não existem duplicidades na chave `RA + ano`;
- os **399 valores de idade de 2023 que haviam sido interpretados como datas** foram recuperados corretamente; após a correção, 2023 não possui idades nulas e apresenta valores entre **7 e 26 anos**;
- gênero foi consolidado em apenas **Feminino** e **Masculino**;
- `Agata` e `Ágata` foram padronizados para **Ágata**;
- os **38 registros `INCLUIR` de 2024** foram tratados como ausência de classificação/INDE, sem excluir os alunos da base;
- após a conversão, o INDE possui **0 nulos em 2022, 83 em 2023 e 102 em 2024**;
- o `IPP` permanece nulo em 2022 por não existir nesse período, evitando imputação artificial;
- todas as fases e fases ideais puderam ser convertidas para uma ordem numérica;
- a relação `Fase Atual - Fase Ideal` reproduz a defasagem original em **100% dos registros de 2022 e 2023** e em **99,83% de 2024**;
- foram identificadas apenas **2 inconsistências de defasagem em 2024** (`RA-1516` e `RA-1519`), preservadas em `defasagem_original` e corrigidas na coluna analítica `defasagem`;
- após a correção, existem **601 alunos defasados em 2022, 552 em 2023 e 534 em 2024**;
- a base temporal 2023 → 2024 possui **765 alunos**;
- nesse conjunto, **308 alunos estão defasados em 2024** e **457 não estão**, oferecendo uma distribuição inicial adequada para testar modelos de classificação;
- entre os **370 alunos que não estavam defasados em 2023**, **84 passaram a ficar defasados em 2024**. Esse segundo alvo poderá ser explorado para uma abordagem mais estrita de “entrada em risco”.

### Decisões metodológicas

Nenhum valor ausente dos indicadores foi imputado neste notebook. Essa decisão é proposital: qualquer imputação utilizada pelo modelo deverá ser aprendida **somente com os dados de treino**, dentro de um pipeline, para evitar *data leakage*.

Também não removemos alunos simplesmente por possuírem indicadores ausentes. A estratégia de exclusão ou imputação dependerá da análise específica realizada nos próximos notebooks.

### Próxima etapa

O notebook `03_analise_exploratoria.ipynb` utilizará `base_tratada.csv` para responder às perguntas do Datathon, incluindo:

1. perfil de defasagem e IAN;
2. evolução do desempenho acadêmico;
3. relação entre engajamento, IDA e IPV;
4. coerência entre autoavaliação e desempenho;
5. padrões psicossociais;
6. relação entre IPP e defasagem;
7. fatores associados ao ponto de virada;
8. combinações de indicadores relacionadas ao INDE;
9. evolução dos indicadores e efetividade do programa.

O treinamento do modelo ficará para o notebook `04_modelo_ml.ipynb`.